In [18]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# File paths
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

GEOJSON_PATH = RAW_DIR / "DataMasterPlan2019PlngAreaBoundaryNoSea.geojson"
DEMAND_PATH = PROCESSED_DIR / "demand_surface.csv"
FACILITIES_PATH = PROCESSED_DIR / "facilities_with_planning_area.csv"

DASHBOARD_OUTPUT = PROCESSED_DIR / "planning_areas_dashboard.geojson"

In [19]:
# Cell 2a — Load source files
demand_df = pd.read_csv(DEMAND_PATH)
facilities_df = pd.read_csv(FACILITIES_PATH)
gdf = gpd.read_file(GEOJSON_PATH)

In [20]:
# Cell 2b — Rebuild merged_df
facility_counts = (
    facilities_df
    .groupby('PLN_AREA_N')['facility_id']
    .agg('count')
    .reset_index()
)

merged_df = demand_df.merge(facility_counts, on='PLN_AREA_N', how='left')
merged_df = merged_df.rename(columns={'facility_id': 'facility_count'})
merged_df['facility_count'] = merged_df['facility_count'].fillna(0).astype(int)

merged_df = merged_df.merge(gdf[['PLN_AREA_N', 'geometry']], on='PLN_AREA_N', how='left')

merged_df['pwd_per_facility'] = merged_df.apply(
    lambda row: row['pwd_estimate'] / row['facility_count'] if row['facility_count'] > 0 else None,
    axis=1
)
merged_df['no_facility'] = merged_df['facility_count'] == 0

merged_df = gpd.GeoDataFrame(merged_df, geometry='geometry')

print(merged_df.shape)
merged_df.head()

(55, 6)


,PLN_AREA_N,pwd_estimate,facility_count,geometry,pwd_per_facility,no_facility
0,ANG MO KIO,5507,7,"POLYGON ((103.85721 1.39654, 103.85701 1.39672...",786.714286,False
1,BEDOK,8197,12,"POLYGON ((103.93208 1.30555, 103.93208 1.30555...",683.083333,False
2,BISHAN,2530,3,"POLYGON ((103.84297 1.36429, 103.84228 1.36435...",843.333333,False
3,BOON LAY,0,0,"POLYGON ((103.72042 1.32824, 103.72066 1.32865...",NaN,True
4,BUKIT BATOK,3316,8,"POLYGON ((103.76408 1.37001, 103.76385 1.37036...",414.500000,False


In [23]:
# Cell 3 — Export dashboard GeoJSON
export_df = merged_df.copy()
export_df['pwd_per_facility'] = export_df['pwd_per_facility'].where(
    export_df['pwd_per_facility'].notna(), other=None
)

export_df.to_file(DASHBOARD_OUTPUT, driver='GeoJSON')

print("Export complete")
print("Rows exported:", len(export_df))
print("Output path:", DASHBOARD_OUTPUT)


Export complete
Rows exported: 55
Output path: ..\data\processed\planning_areas_dashboard.geojson


In [27]:
test_df = gpd.read_file(DASHBOARD_OUTPUT)

print(test_df.shape)
print(test_df.dtypes)
test_df.head()

(55, 6)
PLN_AREA_N            object
pwd_estimate           int32
facility_count         int32
pwd_per_facility     float64
no_facility             bool
geometry            geometry
dtype: object


,PLN_AREA_N,pwd_estimate,facility_count,pwd_per_facility,no_facility,geometry
0,ANG MO KIO,5507,7,786.714286,False,"POLYGON ((103.85721 1.39654, 103.85701 1.39672..."
1,BEDOK,8197,12,683.083333,False,"POLYGON ((103.93208 1.30555, 103.93208 1.30555..."
2,BISHAN,2530,3,843.333333,False,"POLYGON ((103.84297 1.36429, 103.84228 1.36435..."
3,BOON LAY,0,0,NaN,True,"POLYGON ((103.72042 1.32824, 103.72066 1.32865..."
4,BUKIT BATOK,3316,8,414.500000,False,"POLYGON ((103.76408 1.37001, 103.76385 1.37036..."
